In [1]:
from typing import Dict, Tuple, Union, Optional
import xarray as xr
import numpy as np
import torch

from wavelet_base import Wavelet2DBaseTorch
from metrics import Metric

%load_ext autoreload
%autoreload 2

In [2]:
class WaveSim(Wavelet2DBaseTorch, Metric):
    """
    WaveSim class that inherits from Wavelet2DBaseTorch and Metric.
    This class is designed to handle wavelet transformations and metric calculations.
    """

    def __init__(self, 
                 map1: Union[torch.Tensor, xr.DataArray, np.ndarray], 
                 map2: Union[torch.Tensor, xr.DataArray, np.ndarray],
                 params: Dict):
        """
        Initialize the WaveSim class with two maps, parameters, wavelet object, mode, and levels.
        """
        # Initialize Metric (with both maps)
        Metric.__init__(self, map1, map2, params)
        
        self.B, self.C, self.H, self.W = self.map1.shape
        
        self.eps = 1e-8  # Small value to avoid division by zero
        
        # wavelet params (TODO: use get from params)
        self.wavelet = params['wavelet']
        self.mode = params['mode']
        self.levels = params['levels']
        self.operation = params['operation']
        self.components_weight = params['components_weight']
        self.alpha = self.components_weight['alpha']                # (magnitude component weight)
        self.beta = self.components_weight['beta']                  # (displacement component weight)
        self.gamma = self.components_weight['gamma']                # (structural component weight)
        self.use_approx = params.get("use_approx", True)            # Whether to use only detail coefficients

        scales_weight = torch.Tensor(params['scales_weight'])
        expected_len = self.levels + 1 if self.use_approx else self.levels
        
        if scales_weight.shape[-1] != expected_len:
            raise ValueError(
                f"scales_weight tensor last dim mismatch: got {scales_weight.shape[-1]}, "
                f"expected {expected_len} (use_approx={self.use_approx})."
            )
        self.scales_weight_tensor = scales_weight.view(1, 1, expected_len)  # Shape (1, 1, L+1) or (1, 1, L)
        
        #scales_weight = params['scales_weight']
        #self.scales_weight_tensor = torch.Tensor(scales_weight) if isinstance(scales_weight, (list, np.ndarray)) else scales_weight
        #self.scales_weight_tensor = self.scales_weight_tensor.view(self.B, self.C, self.levels + 1) if self.use_approx else self.scales_weight_tensor.view(self.B, self.C, self.levels) 
        
        # Initialize Wavelet2DBaseTorch with the maps
        # compute DWT for both maps according to the wavelet, mode, and levels
        self.map1_dwt = Wavelet2DBaseTorch(data=self.map1, wavelet=self.wavelet, mode=self.mode, levels=self.levels)
        self.map2_dwt = Wavelet2DBaseTorch(data=self.map2, wavelet=self.wavelet, mode=self.mode, levels=self.levels)

    def _magnitude_component(self, m1_energy: torch.Tensor, m2_energy: torch.Tensor) -> torch.Tensor:
        """
        Compute the magnitude component of WaveSim.
        """
        self.m1_mean_energy = torch.mean(m1_energy, dim=(2, 3))                     # (B, C, L+1)
        self.m2_mean_energy = torch.mean(m2_energy, dim=(2, 3))                     # (B, C, L+1)
        sum_energy = self.m1_mean_energy + self.m2_mean_energy                      # (B, C, L+1)   
        magnitude_difference = torch.abs(self.m1_mean_energy - self.m2_mean_energy) # (B, C, L+1)
        relative_difference = magnitude_difference / (sum_energy + self.eps)        # (B, C, L+1)
        return 1 - relative_difference                                              # (B, C, L+1)
    
    def _displacement_component(self, m1_energy: torch.Tensor, m2_energy: torch.Tensor) -> torch.Tensor: 
        """
        Compute the displacement component of WaveSim.
        """
        
        def marginals(energy: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
            """
            Compute the normalized marginals of the energy tensor across latitudes and longitudes.
            """
            total_energy = energy.sum(dim=(-3, -2), keepdim=True)
            lat_marginal = energy.sum(dim=-2) / (total_energy.squeeze(-2) + self.eps)
            lon_marginal = energy.sum(dim=-3) / (total_energy.squeeze(-3) + self.eps)
            return lat_marginal, lon_marginal
        
        def kl_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
            p = torch.clamp(p, min=eps)
            q = torch.clamp(q, min=eps)
            return (p * torch.log2(p / q)).sum(dim=-2)
        
        self.m1_marginal_lat, self.m1_marginal_lon = marginals(m1_energy)           # (B, C, H, L+1), (B, C, W, L+1)
        self.m2_marginal_lat, self.m2_marginal_lon = marginals(m2_energy)           # (B, C, H, L+1), (B, C, W, L+1)
        
        mixture_lat = 0.5 * (self.m1_marginal_lat + self.m2_marginal_lat)           # (B, C, H, L+1)
        mixture_lon = 0.5 * (self.m1_marginal_lon + self.m2_marginal_lon)           # (B, C, W, L+1)
        
        # Compute the Kullback-Leibler divergence for both latitudes and longitudes against the mixture marginals
        # KL(p || m) = sum(p * log(p / m))
        # where p is the marginal distribution of the energy (lat or lon), and m is the mixture marginal
        kl_map1_lat = kl_divergence(self.m1_marginal_lat, mixture_lat)              # (B, C, L+1)
        kl_map2_lat = kl_divergence(self.m2_marginal_lat, mixture_lat)              # (B, C, L+1)
        kl_map1_lon = kl_divergence(self.m1_marginal_lon, mixture_lon)              # (B, C, L+1)
        kl_map2_lon = kl_divergence(self.m2_marginal_lon, mixture_lon)              # (B, C, L+1)
        
        # Compute the Jensen-Shannon divergence for both latitudes and longitudes
        # JSD = 0.5 * (KL(p || m) + KL(q || m))
        # where p and q are the marginals of the two maps, and m is the mixture marginal
        # JSD is symmetric and bounded between 0 and 1
        jsd_lat = 0.5 * (kl_map1_lat + kl_map2_lat)                                 # (B, C, L+1)
        jsd_lon = 0.5 * (kl_map1_lon + kl_map2_lon)                                 # (B, C, L+1)
        return (1 - jsd_lat) * (1 - jsd_lon)
 
    def _structural_component(self, coeffs1: torch.Tensor, coeffs2: torch.Tensor) -> torch.Tensor: 
        """
        Compute the structural component of WaveSim.
        """
        B, C, H, W, S = coeffs1.shape       # S is L+1, the number of scales
        
        # Reshape to (B, C, spatial_dims, S) for processing
        # treat spatial locations as a flat vector of features for each scale S
        c1 = coeffs1.view(B, C, H*W, S)                                             # (B, C, H*W, S)
        c2 = coeffs2.view(B, C, H*W, S)                                             # (B, C, H*W, S)
        
        # Sort along spatial dimension to capture structural patterns
        # This removes spatial positional information, retaining only sorted structural patterns (e.g. edges, textures). 
        # This makes the metric invariant to translations and permutations across the spatial domain.
        c1_sorted, _ = torch.sort(c1, dim=2)                                        # (B, C, H*W, S)
        c2_sorted, _ = torch.sort(c2, dim=2)                                        # (B, C, H*W, S)
        
        # Center the sorted coefficients 
        # Removes global mean and make the component contrast-invariant
        # This ensures we're measuring structural patterns, not absolute levels
        c1_centered = c1_sorted - c1_sorted.mean(dim=2, keepdim=True)               # (B, C, H*W, S)
        c2_centered = c2_sorted - c2_sorted.mean(dim=2, keepdim=True)               # (B, C, H*W, S)     
        
        # L2 normalization to make similarity magnitude-invariant
        # ||x||_2 = sqrt(sum(x^2))
        c1_norm = c1_centered / (c1_centered.norm(dim=2, keepdim=True) + self.eps)  # (B, C, 1, S)
        c2_norm = c2_centered / (c2_centered.norm(dim=2, keepdim=True) + self.eps)  # (B, C, 1, S)

        # Compute normalized dot product (cosine similarity)
        # sim in [-1, 1], then map to [0, 1]
        cosine_sim = (c1_norm * c2_norm).sum(dim=2)                                 # (B, C, S)
        cosine_sim = 0.5 * (cosine_sim + 1.0)  # Map [-1,1] to [0,1]                # (B, C, S)
        
        # Magnitude variation penalty
        #mean_gap = (c1_sorted - c2_sorted).abs().mean(dim=2)  # [0, 2]
        #penalty = 1.0 - mean_gap / (c1_sorted.abs().mean(dim=2) + c2_sorted.abs().mean(dim=2) + self.eps)
        return cosine_sim #* penalty                                                # (B, C, S)
    
    def compute(self) -> float:
        """
        """
        self.m1_coeffs = self.map1_dwt.scale_inversion(operation=self.operation)    # (B, C, H, W, L+1)
        self.m2_coeffs = self.map2_dwt.scale_inversion(operation=self.operation)    # (B, C, H, W, L+1)
        
        if self.use_approx == False:
            self.m1_coeffs = self.m1_coeffs[:,:,:,:,:-1]
            self.m2_coeffs = self.m2_coeffs[:,:,:,:,:-1]
        
        m1_energy = self.m1_coeffs ** 2                                             # (B, C, H, W, L+1)
        m2_energy = self.m2_coeffs ** 2                                             # (B, C, H, W, L+1)       
        
        self.magnitude_sim_score = self._magnitude_component(m1_energy, m2_energy)
        self.magnitude_sim_score_weighted = torch.pow(self.magnitude_sim_score, self.alpha)
       
        self.displacement_sim_score = self._displacement_component(m1_energy, m2_energy)
        self.displacement_sim_score_weighted = torch.pow(self.displacement_sim_score, self.beta)
        
        self.structural_sim_score = self._structural_component(self.m1_coeffs, self.m2_coeffs)
        self.structural_sim_score_weighted = torch.pow(self.structural_sim_score, self.gamma)
        
        self.scale_score = self.magnitude_sim_score_weighted * self.displacement_sim_score_weighted * self.structural_sim_score_weighted        # (B, C, L+1)
        self.score = torch.sum(self.scales_weight_tensor * self.scale_score, dim=-1).item()                                                     # (B, C)

        return self.score
    

In [8]:
params = {'wavelet': 'db1',
        'mode': 'zero',
        'levels': 3,
        'operation': 'sum',
        'components_weight': {'alpha': 1.0, 'beta': 1.0, 'gamma': 1.0},
        'scales_weight': [0.33, 0.33, 0.33], # TODO : transform in a tensor (B, C, L+1)
        'use_approx': False,
        }

wavesim = WaveSim(map1=np.random.random((1, 10, 10)), 
                  map2=np.random.random((1, 10, 10)), 
                  params=params)

In [9]:
wavesim.compute()

0.7652015686035156

In [96]:
dati = torch.Tensor(np.random.random((1, 1, 10, 10)))

In [97]:
wavelet = Wavelet2DBaseTorch(dati, wavelet='db1', mode='zero', levels=3)

In [98]:
scale_inversion = wavelet.scale_inversion(operation='sum')
scale_inversion_ori = wavelet.scale_inversion_ori()
print(scale_inversion.shape, scale_inversion_ori.shape)

torch.Size([1, 1, 10, 10, 4]) torch.Size([1, 1, 10, 10, 4])


In [101]:
(scale_inversion[0,0,:,:,0] - scale_inversion_ori[0,0,:,:,0]).sum()

tensor(-3.7253e-08)

In [80]:
wavesim.displacement_sim_score

tensor([[[0.9271, 0.9341, 0.9402, 0.9986]]])

In [72]:
wavesim.m1_marginal_lat

tensor([[[[0.1200, 0.1250, 0.0895, 0.1219],
          [0.1181, 0.1250, 0.0895, 0.1219],
          [0.1721, 0.0192, 0.0895, 0.1219],
          [0.1001, 0.0192, 0.0895, 0.1219],
          [0.0560, 0.0486, 0.0490, 0.1219],
          [0.0554, 0.0486, 0.0490, 0.1219],
          [0.1889, 0.0485, 0.0490, 0.1219],
          [0.0397, 0.0485, 0.0490, 0.1219],
          [0.0727, 0.2587, 0.2231, 0.0125],
          [0.0770, 0.2587, 0.2231, 0.0125]],

         [[0.1325, 0.0836, 0.0728, 0.1226],
          [0.1206, 0.0836, 0.0728, 0.1226],
          [0.0447, 0.0700, 0.0728, 0.1226],
          [0.1007, 0.0700, 0.0728, 0.1226],
          [0.1131, 0.0947, 0.0686, 0.1226],
          [0.0573, 0.0947, 0.0686, 0.1226],
          [0.0581, 0.0838, 0.0686, 0.1226],
          [0.1202, 0.0838, 0.0686, 0.1226],
          [0.1384, 0.1679, 0.2172, 0.0094],
          [0.1144, 0.1679, 0.2172, 0.0094]],

         [[0.1043, 0.1522, 0.1054, 0.1241],
          [0.0500, 0.1522, 0.1054, 0.1241],
          [0.1284, 0.0917, 0

In [33]:
x = np.random.random((10,10)).shape

In [35]:
len(x)

2

In [25]:
wavesim.map1

array([[0.05998898, 0.09686228, 0.35109839, 0.45114259, 0.3130114 ,
        0.29798441, 0.94241141, 0.40506311, 0.73639726, 0.17194066],
       [0.0267393 , 0.44991165, 0.60434399, 0.18368007, 0.57040981,
        0.04685901, 0.95755789, 0.89166451, 0.216473  , 0.67899538],
       [0.1663537 , 0.3538906 , 0.18958676, 0.70613685, 0.64247217,
        0.29730683, 0.86661721, 0.25512385, 0.55347711, 0.93571734],
       [0.95854346, 0.17091834, 0.2779207 , 0.90412968, 0.29635104,
        0.78551557, 0.14171358, 0.89872823, 0.19102611, 0.82603397],
       [0.90828285, 0.98032409, 0.13059499, 0.78746153, 0.05255254,
        0.35087326, 0.15438634, 0.40783492, 0.93743408, 0.21056637],
       [0.7210267 , 0.08683453, 0.75700471, 0.2389449 , 0.46251787,
        0.29043305, 0.64567875, 0.66965966, 0.70037012, 0.58647035],
       [0.67445131, 0.69214167, 0.26355261, 0.44942742, 0.1714734 ,
        0.34203475, 0.54560309, 0.41339572, 0.59677243, 0.57625615],
       [0.40916054, 0.45659   , 0.6773471